In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.metrics import ConfusionMatrixDisplay # Kept for potential future use of 2x2 matrix
from collections import Counter
import gradio as gr

# ========== 0. 固定 random seed ==========/
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

os.makedirs('./data', exist_ok=True) # 確保data資料夾存在

# ========== 1. 讀取原始資料 ==========
def load_original_data():
    try:
        winners_df = pd.read_csv('./data/winners.csv', encoding='utf-8')
        drivers_df = pd.read_csv('./data/drivers_updated.csv', encoding='utf-8')
        # teams_df = pd.read_csv('./data/teams_updated.csv', encoding='utf-8') # 未在原始特徵中使用
    except FileNotFoundError as e:
        print(f"錯誤：找不到必要的CSV檔案。請確保 './data/' 目錄下有 winners.csv 和 drivers_updated.csv。")
        print(f"詳細錯誤: {e}")
        return None, None

    # 清理字串並轉換基本類型
    for df_ in [winners_df, drivers_df]:
        if df_ is not None:
            for col in df_.columns:
                if df_[col].dtype == 'object':
                    df_[col] = df_[col].astype(str).str.strip()

    if winners_df is not None and 'Date' in winners_df.columns:
        winners_df['year'] = pd.to_datetime(winners_df['Date'], errors='coerce').dt.year
        winners_df = winners_df.dropna(subset=['year']) # 移除無法解析年份的行
        winners_df['year'] = winners_df['year'].astype(int)
    else:
        print("警告：winners.csv 中缺少 'Date' 欄位或無法轉換年份。")
        return None, None


    if drivers_df is not None and 'year' in drivers_df.columns:
        # 嘗試將 'year' 轉換為整數，處理潛在的非數字值
        drivers_df['year'] = pd.to_numeric(drivers_df['year'], errors='coerce')
        drivers_df = drivers_df.dropna(subset=['year']) # 移除無法轉換年份的行
        drivers_df['year'] = drivers_df['year'].astype(int)
    else:
        print("警告：drivers_updated.csv 中缺少 'year' 欄位或 'year' 欄位格式不正確。")
        # 如果 drivers_df 缺失年份，則無法繼續，因為後續處理依賴於此
        return winners_df, None


    # 重命名 drivers_df 中的 'Car' 欄位以匹配 'Team' 的概念
    if drivers_df is not None and 'Car' in drivers_df.columns:
        drivers_df = drivers_df.rename(columns={'Car': 'Team'})
    elif drivers_df is not None:
        print("警告: drivers_updated.csv 中缺少 'Car' 欄位，該欄位將被視為 'Team'。")
        # 如果缺少 'Car'，後續可能出錯，這裡先不填充，讓錯誤在後面暴露

    return winners_df, drivers_df

# ========== 2. 重構數據集：為每個參賽者創建樣本 ==========
def restructure_data(winners_df, drivers_df):
    if winners_df is None or drivers_df is None:
        print("錯誤：由於原始數據加載失敗，無法重構數據集。")
        return None

    required_winner_cols = ['year', 'Grand Prix', 'Winner']
    if not all(col in winners_df.columns for col in required_winner_cols):
        print(f"錯誤：winners_df 缺少必要欄位。需要：{required_winner_cols}")
        return None

    required_driver_cols = ['year', 'Driver', 'Team', 'Nationality']
    if not all(col in drivers_df.columns for col in required_driver_cols):
        print(f"錯誤：drivers_df 缺少必要欄位。需要：{required_driver_cols}")
        return None

    all_samples = []
    # 按年份對 drivers_df 進行分組，以提高查找效率
    drivers_grouped_by_year = {year: group for year, group in drivers_df.groupby('year')}

    for _, race in winners_df.iterrows():
        current_year = race['year']
        current_gp = race['Grand Prix']
        actual_winner_name = race['Winner']

        if current_year not in drivers_grouped_by_year:
            # print(f"警告：在 drivers_df 中找不到年份 {current_year} 的車手數據，跳過比賽：{current_gp} {current_year}")
            continue

        year_drivers_df = drivers_grouped_by_year[current_year]

        for _, participant in year_drivers_df.iterrows():
            sample = {
                'year': current_year,
                'Grand Prix': current_gp,
                'Driver': participant['Driver'],
                'Team': participant['Team'],
                'Nationality': participant['Nationality'],
                'is_winner': 1 if participant['Driver'] == actual_winner_name else 0
            }
            all_samples.append(sample)

    if not all_samples:
        print("錯誤：未能生成任何樣本。請檢查輸入數據和重構邏輯。")
        return None
        
    return pd.DataFrame(all_samples)

# 執行數據加載和重構
original_winners_df, original_drivers_df = load_original_data()
df = None
if original_winners_df is not None and original_drivers_df is not None:
    df = restructure_data(original_winners_df, original_drivers_df)

if df is None:
    print("無法繼續執行，因為數據準備階段發生錯誤。")
    exit()
elif df.empty:
    print("生成的 DataFrame 為空，無法繼續。請檢查CSV文件內容和數據處理邏輯。")
    exit()

# ========== 3. 分組，避免資料洩漏 ==========/
# 確保 'year' 和 'Grand Prix' 是字串類型用於 race_id 生成
df['year_str'] = df['year'].astype(str)
df['gp_str'] = df['Grand Prix'].astype(str)
df['race_id'] = df['year_str'] + "_" + df['gp_str']

unique_race_ids = df['race_id'].unique()
train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=SEED)

train_df = df[df['race_id'].isin(train_ids)].reset_index(drop=True)
test_df = df[df['race_id'].isin(test_ids)].reset_index(drop=True)

# ========== 4. Encoder/Scaler fit only on train ==========
cat_cols = ['Grand Prix', 'Team', 'Driver', 'Nationality']
num_cols = ['year'] # 'year' 本身是數值型
target_col = 'is_winner' # 新的目標欄位

def safe_transform(encoder, data_series):
    # 對於測試集中的未知標籤，賦予一個特殊值（例如 -1 或 0），模型 Embedding 層需要能處理
    # 或者，在 fit 時收集所有可能的值（如果適用）
    # 這裡我們假設測試集的值如果未在訓練集出現，則賦予一個預設編碼值 (e.g., 0, 代表 "unknown")
    # 或者在 LabelEncoder 初始化時用 handle_unknown='use_encoded_value', unknown_value=-1 (需要 scikit-learn 0.24+)
    
    # 簡化處理：將不在 encoder.classes_ 中的值視為一個新類別（索引為0）
    # 注意：這要求 Embedding 層的 num_embeddings 比 len(encoder.classes_) 大至少1，或者特殊處理索引0
    # 更好的做法是確保所有類別都被學習到，或用更高級的處理方式
    
    # 此處的 safe_transform 邏輯調整：如果值不在已知類別中，返回一個代表 "未知" 的固定索引 (例如 0)
    # 這要求在定義 Embedding 層時，`num_embeddings` 要考慮到這個 "未知" 類別
    # 或者，我們可以嘗試將未知的標籤添加到 classes_ 中（這不是標準做法，且只對本地 encoder 有效）
    
    encoded_values = []
    # 建立一個 class 到 index 的映射，並將未知標籤映射到 0
    class_to_index = {cls: i for i, cls in enumerate(encoder.classes_)}
    unknown_index = 0 # 通常我們會將 index 0 保留給 padding 或 unknown
                      # 如果 LabelEncoder 從 0 開始編碼，那麼 0 可能已經被一個已知類別佔用
                      # 一個更穩健的方法是將已知類別的索引+1，0 用作未知
    
    # 修正：讓 LabelEncoder 從1開始編碼，0作為未知
    # 不，LabelEncoder 預設從0開始。我們假設第0個embedding可以代表 "未知"
    
    for x in data_series:
        if x in encoder.classes_:
            encoded_values.append(encoder.transform([x])[0])
        else:
            encoded_values.append(unknown_index) #  假設0是 "unknown"
    return pd.Series(encoded_values, index=data_series.index)


label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # Fit on training data only
    le.fit(train_df[col].astype(str)) # 確保是字串類型
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = safe_transform(le, test_df[col].astype(str)) # 使用 safe_transform
    label_encoders[col] = le

scaler = StandardScaler()
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols] = scaler.transform(test_df[num_cols])

# Target column is already 0 or 1

# ========== 5. Dataset & DataLoader ==========
class F1Dataset(Dataset):
    def __init__(self, df_data, cat_cols, num_cols, target_col):
        self.cat = df_data[cat_cols].values
        self.num = df_data[num_cols].values
        self.y = df_data[target_col].values

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # 確保類別數據中的負數（可能由 safe_transform 引入的未知標籤）被處理
        # 這裡的 safe_transform 已改為返回0，所以不需要特別處理負數
        # cat_features = np.where(self.cat[idx] < 0, 0, self.cat[idx]) # 如果 safe_transform 用 -1
        cat_features = self.cat[idx]
        
        return torch.tensor(cat_features, dtype=torch.long), \
               torch.tensor(self.num[idx], dtype=torch.float32), \
               torch.tensor(self.y[idx], dtype=torch.float32) # BCEWithLogitsLoss 需要 float target

batch_size = 256 # 由於樣本數增加，可以考慮增大 batch_size
trainset = F1Dataset(train_df, cat_cols, num_cols, target_col)
testset = F1Dataset(test_df, cat_cols, num_cols, target_col)

# WeightedRandomSampler for binary classification
class_counts = train_df[target_col].value_counts().to_dict()
if 0 not in class_counts: class_counts[0] = 1 # 避免除以零，如果某類別不存在
if 1 not in class_counts: class_counts[1] = 1

weight_class_0 = 1. / class_counts[0]
weight_class_1 = 1. / class_counts[1]

samples_weight = np.array([weight_class_1 if t == 1 else weight_class_0 for t in train_df[target_col]])
samples_weight = torch.from_numpy(samples_weight).double() # .double() for WeightedRandomSampler
sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

trainloader = DataLoader(trainset, batch_size=batch_size, sampler=sampler)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False)

# ========== 6. DNN+Embedding 模型 (調整為二元分類) ==========
class F1DNN(nn.Module):
    def __init__(self, cat_dims_dict, num_features_numerical, emb_dim=32, hidden_dim=256): # num_classes 改為1
        super().__init__()
        self.embeddings = nn.ModuleList()
        # 為每個類別特徵創建 Embedding 層
        # cat_dims_dict 是一個字典 {'col_name': num_unique_values}
        # 確保 num_embeddings 至少是 max_encoded_value + 1
        # 如果 safe_transform 將未知標籤映射到0，並且0也是一個有效編碼，
        # 那麼 num_embeddings 應該是 len(le.classes_) (如果0確實是有效編碼之一)
        # 或者 len(le.classes_) + 1 (如果0專門給未知，而其他編碼從1開始)
        # 這裡的 LabelEncoder 從0開始編碼已知類別，safe_transform 也將未知映射到0。
        # 這意味著 "未知" 和 "第一個已知類別" 可能共享索引0。
        # 一個更安全的做法是讓 Embedding 層的維度比 le.classes_ 的數量多1，並將未知映射到這個額外索引。
        # 或者，在 safe_transform 中將未知類別映射到 len(le.classes_)
        # 為簡單起見，假設 le.classes_ 包含所有重要類別，未知類別由索引0代表，而0也可能是一個已知類別的編碼
        
        total_emb_output_dim = 0
        for col_name, num_unique_values in cat_dims_dict.items():
            # num_embeddings 應該是該特徵的最大編碼值 + 1
            # 或者更安全的是 train_df[col].max() + 1 (如果已編碼)
            # 或者 len(label_encoders[col_name].classes_)
            # 如果 safe_transform 將未知值映射到0，而0也是一個有效類別，這裡沒問題
            # 如果 LabelEncoder 編碼的值中不包含0（不太可能），則需要調整
            self.embeddings.append(nn.Embedding(num_unique_values, emb_dim))
            total_emb_output_dim += emb_dim

        self.bn_num = nn.BatchNorm1d(num_features_numerical)
        # self.fc1 = nn.Linear(len(cat_cols)*emb_dim + num_features_numerical, hidden_dim) # 原來的寫法
        self.fc1 = nn.Linear(total_emb_output_dim + num_features_numerical, hidden_dim)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5) # 可以調整 dropout rate
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, 1) # 輸出1個 logit

    def forward(self, x_cat, x_num):
        # x_cat 的 shape: (batch_size, num_categorical_features)
        x_emb_list = []
        for i, emb_layer in enumerate(self.embeddings):
            x_emb_list.append(emb_layer(x_cat[:, i]))
        
        x_emb = torch.cat(x_emb_list, dim=1)
        
        if x_num.ndim == 1: # 如果 num_features_numerical 是 1，x_num 可能是一維的
             x_num = x_num.unsqueeze(1)
        if x_num.shape[1] > 0: # 只有當有數值特徵時才使用bn
            x_num = self.bn_num(x_num)
            x = torch.cat([x_emb, x_num], dim=1)
        else:
            x = x_emb # 如果沒有數值特徵

        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x) # 輸出 logits
        return x

# cat_dims: 每個類別特徵的唯一值數量 (基於訓練集 LabelEncoder)
cat_dims_dict = {col: len(le.classes_) for col, le in label_encoders.items()}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = F1DNN(cat_dims_dict, len(num_cols), emb_dim=32, hidden_dim=256).to(device)

# ========== 7. 訓練（記錄loss曲線）==========
def train_model(model, trainloader, testloader, n_epoch=50, lr=0.001, patience=7): # 調整超參數
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5) # 加入 weight_decay
    criterion = nn.BCEWithLogitsLoss() # 用於二元分類
    
    best_test_loss = np.inf # 基於測試損失進行早停
    no_improve = 0
    train_losses = []
    test_losses = []

    for epoch in range(n_epoch):
        model.train()
        total_train_loss = 0
        for x_cat, x_num, y in trainloader:
            x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
            
            optimizer.zero_grad()
            logits = model(x_cat, x_num)
            loss = criterion(logits, y.unsqueeze(1)) # y 需要是 (batch_size, 1)
            
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(trainloader)
        train_losses.append(avg_train_loss)

        # Test loss
        model.eval()
        total_test_loss = 0
        with torch.no_grad():
            for x_cat, x_num, y in testloader:
                x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
                logits = model(x_cat, x_num)
                loss = criterion(logits, y.unsqueeze(1))
                total_test_loss += loss.item()
        
        avg_test_loss = total_test_loss / len(testloader)
        test_losses.append(avg_test_loss)
        
        print(f"Epoch {epoch+1}/{n_epoch} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")

        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            no_improve = 0
            torch.save(model.state_dict(), './data/f1_dnn_binary_corrected.pth')
            # print(f"Best model saved with test loss: {best_test_loss:.4f}")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping triggered after {patience} epochs without improvement on test loss.")
                break
    
    # 畫 loss 曲線
    if train_losses and test_losses: # 確保列表不為空
        plt.figure(figsize=(7,5))
        plt.plot(train_losses, label='Train Loss')
        plt.plot(test_losses, label='Test Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Train vs. Test Loss Curve (Binary Classification)')
        plt.legend()
        plt.tight_layout()
        plt.savefig('./data/loss_curve_binary_corrected.png')
        plt.close()
        print("Loss curve saved to ./data/loss_curve_binary_corrected.png")
    else:
        print("沒有足夠的數據來繪製損失曲線。")

    return train_losses, test_losses

# ========== 8. 驗證 (調整為二元分類) ==========
def eval_model_binary(model, testloader):
    model.eval()
    all_y_true = []
    all_y_pred_probs = [] # 儲存預測概率
    all_y_pred_classes = [] # 儲存預測類別 (0 or 1)

    with torch.no_grad():
        for x_cat, x_num, y in testloader:
            x_cat, x_num = x_cat.to(device), x_num.to(device)
            logits = model(x_cat, x_num)
            probs = torch.sigmoid(logits).cpu().numpy().flatten() # (batch_size,)
            preds = (probs > 0.5).astype(int) # (batch_size,)
            
            all_y_pred_probs.extend(probs)
            all_y_pred_classes.extend(preds)
            all_y_true.extend(y.cpu().numpy().flatten())
            
    if not all_y_true:
        print("測試集為空或未能處理，無法進行評估。")
        return 0.0

    acc = accuracy_score(all_y_true, all_y_pred_classes)
    print(f"\nTest Accuracy: {acc:.4f}")
    
    # 確保 labels 參數只包含實際出現在 y_true 和 y_pred 中的值
    unique_labels_in_data = np.unique(np.concatenate((all_y_true, all_y_pred_classes)))
    # 如果預期是0和1，但數據中只有一個類別，classification_report 會出問題
    # target_names=['Not Winner', 'Winner']
    
    # 處理 classification_report 中可能只有一個類別的情況
    # 如果unique_labels_in_data中只有一個值，classification_report會報錯
    # 我們至少需要兩個標籤才能生成報告，即使其中一個標籤的樣本數為零
    report_labels = sorted(list(set(unique_labels_in_data).union({0,1})))


    try:
        print(classification_report(all_y_true, all_y_pred_classes, labels=report_labels, target_names=[f"Class_{i}" for i in report_labels], zero_division=0))
    except ValueError as e:
        print(f"生成 classification_report 時出錯: {e}")
        print(f"True labels: {np.unique(all_y_true, return_counts=True)}")
        print(f"Predicted classes: {np.unique(all_y_pred_classes, return_counts=True)}")


    # 可以選擇性地繪製 2x2 混淆矩陣
    cm = confusion_matrix(all_y_true, all_y_pred_classes, labels=report_labels)
    print("\nConfusion Matrix (Rows: True, Cols: Predicted):")
    print(f"Labels: {report_labels}")
    print(cm)
    # disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Winner', 'Winner'])
    # fig, ax = plt.subplots(figsize=(6, 6))
    # disp.plot(ax=ax, cmap='Blues', colorbar=True)
    # plt.title("Confusion Matrix (Test Set - Binary)")
    # plt.tight_layout()
    # plt.savefig('./data/confusion_matrix_binary_corrected.png')
    # plt.close()
    # print("Confusion matrix saved to ./data/confusion_matrix_binary_corrected.png")

    return acc

# ========== 9. 訓練＆驗證 ==========/
print("開始訓練模型...")
train_losses, test_losses = train_model(model, trainloader, testloader, n_epoch=30, lr=5e-4, patience=5) # 減少 epoch 和 patience 以便快速檢查

# 加載最佳模型進行評估
if os.path.exists('./data/f1_dnn_binary_corrected.pth'):
    model.load_state_dict(torch.load('./data/f1_dnn_binary_corrected.pth', map_location=device))
    print("\n加載已保存的最佳模型進行最終評估...")
    eval_model_binary(model, testloader)
else:
    print("警告：未找到已保存的模型權重。評估將使用訓練結束時的模型狀態。")
    eval_model_binary(model, testloader)


# ========== 10. Gradio 預測邏輯準備 ==========
# 需要 original_drivers_df 來獲取特定年份的車手及其車隊和國籍
# 這個字典應該在 Gradio 啟動前準備好
driver_info_by_year = {}
if original_drivers_df is not None:
    for year_val, group in original_drivers_df.groupby('year'):
        driver_info_by_year[year_val] = []
        for _, row in group.iterrows():
            # 確保 'Driver', 'Team', 'Nationality' 欄位存在
            driver_name = row.get('Driver', 'Unknown Driver')
            team_name = row.get('Team', 'Unknown Team') # 'Team' 是重命名後的 'Car'
            nationality_val = row.get('Nationality', 'Unknown Nationality')
            driver_info_by_year[year_val].append({
                'Driver': driver_name,
                'Team': team_name,
                'Nationality': nationality_val
            })
else:
    print("警告：original_drivers_df 為空，Gradio 的車手列表將不完整。")


all_available_years_for_gradio = sorted(list(driver_info_by_year.keys())) if driver_info_by_year else [2023] # 提供預設值
all_available_grandprix_for_gradio = []
if original_winners_df is not None and 'Grand Prix' in original_winners_df.columns:
     all_available_grandprix_for_gradio = sorted(original_winners_df['Grand Prix'].astype(str).unique())
else:
     all_available_grandprix_for_gradio = ["Monaco Grand Prix"] # 提供預設值


# ========== 11. Gradio預測函數 (調整為二元分類) ==========/
def gradio_predict(year_input, grand_prix_input):
    try:
        year = int(year_input)
    except ValueError:
        return "錯誤：年份輸入無效。"

    if not driver_info_by_year or year not in driver_info_by_year:
        return f"該年份 ({year}) 查無參賽車手資料！請檢查 drivers_updated.csv。"

    participants_for_year = driver_info_by_year[year]
    if not participants_for_year:
        return f"年份 {year} 的參賽車手列表為空。"

    all_results = []
    
    model.eval() # 確保模型在評估模式

    for participant_info in participants_for_year:
        driver_name = participant_info['Driver']
        team_name = participant_info['Team']
        nationality = participant_info['Nationality']

        # 準備模型輸入
        input_dict_cat = {
            'Grand Prix': grand_prix_input,
            'Team': team_name,
            'Driver': driver_name,
            'Nationality': nationality
        }
        
        x_cat_input_encoded = []
        for col_idx, col_name in enumerate(cat_cols):
            le = label_encoders[col_name]
            val_to_encode = input_dict_cat[col_name]
            if val_to_encode in le.classes_:
                x_cat_input_encoded.append(le.transform([val_to_encode])[0])
            else:
                x_cat_input_encoded.append(0) # 未知標籤使用索引0 (需要與訓練時一致)
        
        x_cat_tensor = torch.tensor([x_cat_input_encoded], dtype=torch.long).to(device)
        
        # 數值特徵
        # 創建一個臨時的 DataFrame 來使用 scaler
        # 注意：scaler.transform 期望輸入是2D array-like
        year_scaled = scaler.transform(np.array([[float(year)]]))[0] # scaler expects 2D, takes first row
        x_num_tensor = torch.tensor([year_scaled], dtype=torch.float32).to(device)
        if x_num_tensor.ndim == 1 and len(num_cols) > 0 : # 確保是2D (batch, num_features)
            x_num_tensor = x_num_tensor.unsqueeze(0)
        elif len(num_cols) == 0: # 如果沒有數值特徵
            x_num_tensor = torch.empty(x_cat_tensor.shape[0], 0, dtype=torch.float32).to(device)


        with torch.no_grad():
            logits = model(x_cat_tensor, x_num_tensor)
            # print(f"Driver: {driver_name}, Logits: {logits.item()}") # Debug
            probability = torch.sigmoid(logits).cpu().item() # 單個概率值
        
        all_results.append((driver_name, team_name, probability))

    if not all_results:
        return "未能對任何車手進行預測。"

    all_results.sort(key=lambda x: x[2], reverse=True) # 按概率降序排列

    text_output = f"年份：{year}  場地：{grand_prix_input}\n\n預測冠軍機率排行（前5名）：\n\n"
    for i, (d_name, t_name, prob) in enumerate(all_results[:5], 1):
        text_output += f"{i}. {d_name} ({t_name})：{prob:.2%}\n"
    
    # 可以在這裡加入之前對極高概率的警告
    if all_results and all_results[0][2] > 0.98 and len(all_results) > 1 : # 避免只有一個結果時也警告
         text_output += "\n⚠️ 機率極高可能代表此組合在訓練數據中非常強勢，或模型對此組合過度自信，結果僅供參考。\n"
    if all_results and all_results[0][2] < 0.05 : # 如果最高機率都很低
         text_output += "\nℹ️ 所有車手預測獲勝機率均較低，賽果可能較難預測或缺乏足夠的相關訓練數據。\n"

    return text_output

# ========== 12. Gradio UI ==========/
if not all_available_years_for_gradio: all_available_years_for_gradio.append(2023) # Fallback
if not all_available_grandprix_for_gradio: all_available_grandprix_for_gradio.append("Unknown GP") # Fallback

with gr.Blocks() as demo:
    gr.Markdown("## F1 冠軍預測互動系統 (修正資料洩漏版 - 二元分類)")
    
    with gr.Tab("冠軍預測"):
        year_dropdown = gr.Dropdown(label="年份", choices=all_available_years_for_gradio, value=all_available_years_for_gradio[-1])
        gp_dropdown = gr.Dropdown(label="Grand Prix 場地", choices=all_available_grandprix_for_gradio, value=all_available_grandprix_for_gradio[0])
        predict_btn = gr.Button("預測該場冠軍機率")
        output_textbox = gr.Textbox(label="預測結果", lines=10)
        
        predict_btn.click(
            gradio_predict,
            inputs=[year_dropdown, gp_dropdown],
            outputs=output_textbox
        )
        
    with gr.Tab("訓練/測試 Loss 曲線"):
        gr.Markdown("以下為訓練過程 loss 曲線（訓練結束自動生成）：")
        # 檢查圖片是否存在
        loss_curve_path = "./data/loss_curve_binary_corrected.png"
        if os.path.exists(loss_curve_path):
            gr.Image(value=loss_curve_path, label="Loss Curve")
        else:
            gr.Markdown(f"Loss curve 圖片 ({loss_curve_path}) 未找到。請先完成模型訓練。")
            
    # 移除了 Top-10 混淆矩陣，因為它不適用於二元分類目標
    # 如果需要，可以後續添加一個簡單的 2x2 混淆矩陣圖片展示
    # with gr.Tab("測試集混淆矩陣 (二元)"):
    #     gr.Markdown("以下為測試集二元分類混淆矩陣（訓練結束自動生成）：")
    #     cm_path = "./data/confusion_matrix_binary_corrected.png"
    #     if os.path.exists(cm_path):
    #         gr.Image(value=cm_path, label="Confusion Matrix")
    #     else:
    #         gr.Markdown(f"Confusion matrix 圖片 ({cm_path}) 未找到。")

print("啟動 Gradio 界面...")
demo.launch()